In [1]:
!pip install quimb tqdm scipy
!pip install kahypar optuna
import torch
import quimb as qu
import quimb.tensor as qtn
import numpy as np
import scipy.linalg
from tqdm import tqdm
import time
import random
import os
from contextlib import contextmanager

# ==============================================================================
# 0. DEBUG TOOLS & SYSTEM CHECK
# ==============================================================================

@contextmanager
def CodeTimer(label):
    """Simple timer to measure code blocks with GPU sync."""
    if torch.cuda.is_available():
        torch.cuda.synchronize() # Wait for GPU to finish before starting timer
    t0 = time.time()
    yield
    if torch.cuda.is_available():
        torch.cuda.synchronize() # Wait for GPU to finish before stopping timer
    t1 = time.time()
    print(f"  [TIMER] {label}: {t1 - t0:.5f} sec")

print("\n=== SYSTEM DIAGNOSTICS ===")
try:
    print("--- Shell Check (nvidia-smi) ---")
    if os.system('nvidia-smi') != 0:
        print("    [!] nvidia-smi failed or not found.")
except Exception as e:
    print(f"    [!] Error running nvidia-smi: {e}")

print(f"\n--- PyTorch Check ---")
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("    [!] No GPU detected by PyTorch.")
print("==========================\n")

# ==============================================================================
# 1. Hardware Configuration (Interactive)
# ==============================================================================
print(f"System Check: PyTorch {torch.__version__}")
if torch.cuda.is_available():
    print(f"  [+] NVIDIA GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("  [-] No NVIDIA GPU Detected.")

# --- Interactive Prompt ---
while True:
    choice = input("\nDo you want to run on GPU? (y/n): ").strip().lower()
    if choice in ['y', 'yes']:
        want_gpu = True
        break
    elif choice in ['n', 'no']:
        want_gpu = False
        break

# --- Device Logic ---
if want_gpu:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        target_dtype = torch.complex64  # Double precision is fast on NVIDIA
        print("\n>>> CONFIG: Using NVIDIA CUDA acceleration.")
    else:
        print("\n>>> WARNING: GPU requested but not available. Falling back to CPU.")
        device = torch.device("cpu")
        target_dtype = torch.complex128
else:
    print("\n>>> CONFIG: Using CPU (User Requested).")
    device = torch.device("cpu")
    target_dtype = torch.complex128

# Set Quimb to use PyTorch
qtn.set_tensor_linop_backend('torch')

def to_device(x):
    """Moves data to selected device with correct precision."""
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=target_dtype)
    return torch.tensor(x, dtype=target_dtype, device=device)

# This code is written for 1D TFIM. If not TFIM model, Parts 1, 2, 5 are still compatible as long as in 1D qubit system.
# Pauli Setup
XI = qu.pauli('X') & qu.pauli('I'); IX = qu.pauli('I') & qu.pauli('X')
YI = qu.pauli('Y') & qu.pauli('I'); IY = qu.pauli('I') & qu.pauli('Y')
IZ = qu.pauli('I') & qu.pauli('Z'); ZI = qu.pauli('Z') & qu.pauli('I')
Z, Y, X = qu.pauli('Z'), qu.pauli('Y'), qu.pauli('X')
XX = X & X; XZ = X & Z; ZX = Z & X; XY = X & Y; YX = Y & X


=== SYSTEM DIAGNOSTICS ===
--- Shell Check (nvidia-smi) ---
    [!] nvidia-smi failed or not found.

--- PyTorch Check ---
PyTorch Version: 2.9.1
    [!] No GPU detected by PyTorch.

System Check: PyTorch 2.9.1
  [-] No NVIDIA GPU Detected.


sh: nvidia-smi: command not found



>>> CONFIG: Using CPU (User Requested).


In [2]:
# 1. Insert Functions for MPS
# ==============================================================================

def insert_MPS_site(psi, site_i):
    """
    Inserts the bath AFTER site_i.
    Guarantees the output is an Open Boundary (OBC) MPS (cyclic=False).
    site_i can be -1, meaning the bath is inserted before site 0.
    """
    clean_data = []
    L_old = psi.L

    # 1. Standardize existing tensors to 3D (Left, Right, Phys)
    for i in range(L_old):
        t = psi[i]
        d = t.data
        # Safety: Ensure data is on the correct device/format
        if not isinstance(d, torch.Tensor):
            d = to_device(d)
        # Robustly find the physical axis
        p_axis = t.inds.index(psi.site_ind(i))
        if d.ndim == 2:
            # Handle edge cases where tensors might be rank-2
            if p_axis == 0:
                # d.T in numpy is d.transpose(0, 1) in torch
                d = d.transpose(0, 1).reshape(1, d.shape[1], 4)
            else:
                d = d.reshape(d.shape[0], 1, 4)
        else:
            # Transpose to standard (Left, Right, Phys) layout
            axes = [ax for ax in range(3) if ax != p_axis] + [p_axis]
            # --- FIX: Use .permute(*axes) instead of .transpose(axes) ---
            d = d.permute(*axes).contiguous()
        clean_data.append(d)

    # --- Determine Target Position ---
    target_pos = site_i + 1

    # 2. Determine Bond Dimensions based on target_pos
    if target_pos <= 0:
        l_bond, r_bond = 1, clean_data[0].shape[0]
    elif target_pos >= L_old:
        l_bond, r_bond = clean_data[-1].shape[1], 1
    else:
        l_bond = clean_data[target_pos - 1].shape[1]
        r_bond = clean_data[target_pos].shape[0]

    # 3. Create the "Pass-Through" Bath Tensor
    bath_data = torch.zeros((l_bond, r_bond, 4), dtype=target_dtype, device=device)
    # Set identity-like path
    for b in range(min(l_bond, r_bond)):
        bath_data[b, b, 0] = 1.0

    # 4. Insert into the list
    clean_data.insert(target_pos, bath_data)

    # 5. Rebuild correctly for OBC
    new_psi = qtn.MatrixProductState(clean_data, shape='lrp')
    # Manually ensure the cyclic flag is False and indices are clean
    new_psi.cyclic = False

    # Robust check: if the first and last tensor share an index, rename it.
    t0, tL = new_psi[0], new_psi[new_psi.L - 1]
    shared = set(t0.inds) & set(tL.inds)
    for ind in shared:
        if not ind.startswith('k'):
            tL.reindex_({ind: qtn.rand_uuid()}, inplace=True)

    return new_psi

In [3]:
# 2. Hamiltonian MPO/MPS Setup
# ==============================================================================

def build_sys_hamiltonian(L, pxx, pz):
    builder = qtn.SpinHam1D(S=1/2)
    for i in range(L-1):
        builder[i,i+1] += pxx, X, X
    for i in range(L):
        builder[i] += pz, Z
    return builder.build_mpo(L=L)

def build_hamiltonian_part(L, pxx, pz, omega, alpha, site_i, pauli_op):
    """
    This builds the 1D-TFIM model with an additional bath interacts at sites_{i}.
    The bath is a single qubit appears after site_{i} to ensure 1-local interactions.
    H =  sum_{<i,j>} pxx * X_i X_j  + sum_i pz * Z_i
        + (-omega/2) * Z_bath
        + alpha * S_system * X_bath
    where S_system is one of {X, Y, Z} specified by pauli_op = {1, 2, 3}.
    Note: The bath qubit is indexed as site_{i+1} in the full Hamiltonian.
    Note: site_i can be -1, meaning the bath is inserted before site 0.
    """

    L_tot = L + 1
    bath_idx = site_i + 1
    builder = qtn.SpinHam1D(S=3/2, cyclic=False)

    # --- 1. Define Operators Correctly ---
    # Left operators act on Ket (H rho) -> Use H
    # Right operators act on Bra (-rho H) -> Use H^T

    # Standard Superoperator Basis
    # XI = X tensor I (Acts on Ket)
    # IX = I tensor X (Acts on Bra)
    # Note: For Bra operators (IX, IY, IZ), we must apply the transpose factor of the Pauli.

    # Transpose factors: X->+1, Y->-1, Z->+1

    # System Bonds
    for j in range(site_i):
        # Term: p * S * S
        # Ket side: +p * (S x I) * (S x I)
        # Bra side: -p * (I x S^T) * (I x S^T)
        # If S=Y, S^T = -Y. (-Y)*(-Y) = Y*Y. Signs cancel for pairs!
        # So we don't need to change the bond loops (pyy is safe).
        builder[j, j+1] += pxx, XI, XI; builder[j, j+1] += -pxx, IX, IX

    for j in range(site_i + 1, L - 1):
        u, v = j + 1, j + 2
        builder[u, v] += pxx, XI, XI; builder[u, v] += -pxx, IX, IX

    # Fields (Single operators - These DO need correction for Y)
    for j in range(L):
        idx = j if j <= site_i else j + 1

        # Z Field
        builder[idx] += pz, ZI
        builder[idx] += -pz, IZ # Z^T = Z

    # Bath Field (Z)
    builder[bath_idx] += (-omega/2.0), ZI
    builder[bath_idx] += -(-omega/2.0), IZ

    # Interaction: alpha * S_system * X_bath
    # Ket: +alpha * (S x I) * (X x I) -> OP_L, XI
    # Bra: -alpha * (I x S^T) * (I x X^T) -> OP_R_Transposed, IX

    if pauli_op == 1:   # X
        OP_L = XI
        OP_R = IX # X^T = X
        factor = 1.0
    elif pauli_op == 2: # Y
        OP_L = YI
        OP_R = IY # We use IY, but we multiply by factor -1 because Y^T = -Y
        factor = -1.0
    elif pauli_op == 3: # Z
        OP_L = ZI
        OP_R = IZ # Z^T = Z
        factor = 1.0

    if site_i >= 0: #bath after site_i
        builder[site_i, bath_idx] += alpha, OP_L, XI

        #  Apply the factor for the transpose
        builder[site_i, bath_idx] += -alpha * factor, OP_R, IX
    else: #bath before site 0
        builder[0, 1] += alpha, XI, OP_L

        #  Apply the factor for the transpose
        builder[0, 1] += -alpha * factor, IX, OP_R

    if bath_idx < L_tot - 1:
        builder[bath_idx, bath_idx+1] += 0.0, XI, XI

    H = builder.build_local_ham(L_tot)
    H.cyclic = False
    return H

In [4]:
# 3. Trace out MPS Site Function
# ==============================================================================

def get_vectorized_trace(mps_vec):
    """
    Computes Tr(rho) for a vectorized MPS by contracting with the Identity state.
    Assumes local dimension is 4 (flattened 2x2 density matrix).
    Identity = |00> + |11> -> indices 0 and 3 in flattened basis.
    """
    # Create the Local Identity Tensor for one site
    # Shape (bond_dim_left, physical_dim, bond_dim_right)
    # The Identity vector is [1, 0, 0, 1] for basis [00, 01, 10, 11]
    I_data = torch.tensor([1.0, 0.0, 0.0, 1.0], dtype=mps_vec[0].data.dtype, device=mps_vec[0].data.device)
    I_data = I_data.reshape(1, 4, 1) # Bond dim 1

    # Create the Identity MPS
    arrays = [I_data for _ in range(mps_vec.L)]
    I_mps = qtn.MatrixProductState(arrays, shape='lpr')

    # 3. Contract
    # This returns a Quimb Tensor with shape (1, 1) (the boundary indices)
    trace_qtensor = (I_mps.H @ mps_vec)

    # 4. Extract Scalar
    # .data gives the PyTorch tensor (shape 1x1)
    # .item() extracts the single value as a Python scalar
    return trace_qtensor.data.reshape(-1)[0].item()

def mps_trace_out_site_mpslevel(mps, i, trace_vec=[1, 0, 0, 1]):
    """
    Trace out site i (0 <= i <= L-1) from a density-matrix MPS.

    Handles both bulk/right sites (merging Left) and site 0 (merging Right).
    """
    L = mps.L

    # Validation
    if not (0 <= i <= L - 1):
        raise ValueError(f"i must satisfy 0 <= i <= L-1, got i={i}, L={L}.")

    trace_vec = to_device(trace_vec)

    # --- normalize a tensor to (left, right, phys) using its index name ---
    def tensor_to_lrp(T, phys_ind):
        bonds = [ix for ix in T.inds if ix != phys_ind]
        # Quimb standard: usually [left_bond, right_bond]
        left, right = bonds[0], bonds[1]
        TT = T.transpose(left, right, phys_ind)
        data = TT.data
        if not isinstance(data, torch.Tensor):
            data = to_device(data)
        return data

    # --- extract & normalize all site arrays to (Dl,Dr,4) reliably ---
    Ts = [t.copy() for t in mps.tensors]
    A = []
    for s, T in enumerate(Ts):
        As = tensor_to_lrp(T, f"k{s}")
        A.append(As)

    A_out = []

    # ==========================================================
    # CASE 1: Trace out the first site (i=0)
    # Contract Site 0 -> Matrix M, then M acts on Site 1 from the left.
    # ==========================================================
    if i == 0:
        A_0 = A[0]  # Shape: (1, D_mid, 4)
        A_1 = A[1]  # Shape: (D_mid, D_right, 4)

        # 1. Contract physical index of A_0 with trace_vec
        # Result M shape: (1, D_mid)
        M = torch.tensordot(A_0, trace_vec, dims=([2], [0]))

        # 2. Contract M with A_1
        # M is (1, D_mid), A_1 is (D_mid, D_right, 4)
        # Contract dim 1 of M with dim 0 of A_1
        A_new_0 = torch.tensordot(M, A_1, dims=([1], [0]))

        # Result shape is (1, D_right, 4).
        # tensordot output order: (M_dim0, A1_dim1, A1_dim2) -> (Left, Right, Phys)
        # This matches our required format, so no permutation needed.

        A_out.append(A_new_0)

        # Append the rest of the chain (sites 2 to L-1)
        for s in range(2, L):
            A_out.append(A[s])

    # ==========================================================
    # CASE 2: Trace out bulk or right site (i >= 1)
    # Contract Site i -> Matrix M, then Site i-1 absorbs M from the right.
    # ==========================================================
    else:
        # Contract site i physical index with trace_vec -> matrix M (Dl_i, Dr_i)
        A_im1 = A[i - 1]   # (Dl_{i-1}, Dmid, 4)
        A_i = A[i]         # (Dmid, Dr_i, 4)

        M = torch.tensordot(A_i, trace_vec, dims=([2], [0]))

        # Contract A_{i-1} with M
        # A_im1 is (Dl, Dmid, Phys), M is (Dmid, Dr)
        # Contract dim 1 of A_im1 with dim 0 of M
        A_new_raw = torch.tensordot(A_im1, M, dims=([1], [0]))

        # Result shape: (Dl, Phys, Dr). We need (Dl, Dr, Phys).
        A_new_im1 = A_new_raw.permute(0, 2, 1).contiguous()

        # Build output list
        for s in range(L):
            if s == i - 1:
                A_out.append(A_new_im1)
            elif s == i:
                continue
            else:
                A_out.append(A[s])

    L_out = L - 1

    # --- build tensors explicitly with required inds/tags ---
    Ts_out = []
    for s in range(L_out):
        Ts_out.append(
            qtn.Tensor(
                data=A_out[s],
                inds=(f"b{s}", f"b{s+1}", f"k{s}"),
                tags={f"I{s}"},
            )
        )

    # --- convert TN -> MPS view  ---
    tn = qtn.TensorNetwork(Ts_out)
    out = qtn.MatrixProductState.from_TN(
        tn,
        L=L_out,
        site_ind_id="k{}",
        site_tag_id="I{}",
        cyclic=False,
    )

    # hard-set non-cyclic flags
    try:
        out.cyclic = False
    except Exception:
        pass
    if hasattr(out, "_cyclic"):
        out._cyclic = False
    # print('After trace out_0', get_vectorized_trace(out))
    out /= get_vectorized_trace(out)
    # print('After trace out', get_vectorized_trace(out))
    return out

In [5]:
# 4. MPS Time Evolution acceleration with boundary dissipation
# ============================================================

def run_mpo_step_boundary_TEBD(
    rho, duration, dt, L,
    pxx, pz, omega, alpha,
    site_i, pauli_op, max_bond
):
    # --- 1) setup state ---
    mps = rho.copy()
    mps = insert_MPS_site(mps, site_i)
    mps.cyclic = False
    max_bond_now = mps.max_bond()
    # 1. Build the Hamiltonian (currently contains NumPy arrays)
    H_local = build_hamiltonian_part(L, pxx, pz, omega, alpha, site_i, pauli_op)

    # 2. CONVERT H_local to PyTorch Tensors
    for key in H_local.terms:
    # Convert the numpy array to a torch tensor
        H_local.terms[key] = torch.tensor(
            H_local.terms[key],
            dtype=target_dtype,
            device=device
        )
    # --- 3) options ---
    # if target_dtype == torch.complex64:
    #    split_opts = {'max_bond': max_bond, 'cutoff': 1e-6, 'renorm': True}
    # else:
    #    split_opts = {'max_bond': max_bond, 'cutoff': 1e-12, 'renorm': True}
    split_opts =  {'max_bond': max_bond, 'cutoff': 1e-12, 'cutoff_mode': 'rel', 'renorm': True, 'absorb': 'both'}
    steps = int(duration / dt)
    if steps <= 0:
        return mps, max_bond_now
    tebd = qtn.TEBD(mps, H_local, dt = dt)
    tebd.split_opts = split_opts
    ts = np.linspace(0, duration, steps + 1)[1:]
    for evolved_state in tebd.at_times(ts):
        max_bond_now = max(max_bond_now, evolved_state.max_bond())
    mps = evolved_state
    mps = mps_trace_out_site_mpslevel(mps, site_i + 1)
    curr_trace = get_vectorized_trace(mps)
    # if np.abs(1-curr_trace)>1e-2:
    #     print('Warning! Current trace:', curr_trace)
    mps /= curr_trace
    # if np.abs(1-curr_trace)>1e-2:
    #     print('Warning! Current trace after normalization:', get_vectorized_trace(mps))
    max_bond_now = mps.max_bond()
    return mps, max_bond_now


In [6]:
# 4.5. Single gate evolution code
#=============================================================================

# ----------------------------
# GPU/PyTorch Compatible SWAP Gate
# ----------------------------
def _make_swap_gate_phys4():
    d = 4
    # Initialize as PyTorch Tensor on the Device
    # dtype=complex is required for Apple MPS
    SWAP = torch.zeros((d * d, d * d), dtype=target_dtype, device=device)

    for a in range(d):
        for b in range(d):
            col = a * d + b
            row = b * d + a
            SWAP[row, col] = 1.0

    # Reshape and return
    return SWAP.reshape(d, d, d, d).contiguous()

SWAP_GATE_4 = _make_swap_gate_phys4()



# ----------------------------
# SAFE cached gate generation (uses exact H_local matrices)
# ----------------------------
def generate_hamiltonian_gates_cached(H_local, dt, site_i, expm_cache=None, tol=1e-15):
    """
      - uses the actual matrices from H_local.terms (so SpinHam1D conventions match)
      - caches expm by hashing matrix bytes
      - identity fastpath for zero matrices

    Returns:
      gates: dict {(i,j): gate_tensor} with shape (4,4,4,4)
      expm_cache: dict cache (reuse across calls)
    """
    if expm_cache is None:
        expm_cache = {}

    gates = {}
    for where, term in H_local.terms.items():
        # sanity check (optional)
        # # This TEBD path expects 2-site gates only:
        # if not (isinstance(where, tuple) and len(where) == 2):
        #     continue
        (i, j) = where
        dim_matrix = int(np.sqrt(term.size))
        mat_cpu = term.reshape(dim_matrix, dim_matrix)

        # if mat.shape != (16, 16):
        #     raise ValueError(
        #         f"Unexpected term shape {mat.shape} at where={where}. "
        #         "Expected (16,16) for d=4 two-site superoperator."
        #     )

        # Zero -> identity
        if np.allclose(mat_cpu, 0.0, atol=tol, rtol=0.0):
            # Identity on GPU
            U = torch.eye(16, dtype=target_dtype, device=device)
        else:
            # Cache Key (using CPU bytes)
            key = (round(float(dt), 15), mat_cpu.tobytes())
            U = expm_cache.get(key)
            if U is None:
                if i == site_i or i == site_i + 1:
                    # 1. Move to GPU
                    mat_gpu = to_device(mat_cpu)
                    # 2. Native CUDA eigh
                    w, V = torch.linalg.eigh(mat_gpu)
                    # 3. Exponentiate
                    U = V @ torch.diag(torch.exp(-1j * w * dt)) @ V.mH

                else:
                    # 1. Move to GPU
                    mat_gpu = to_device(mat_cpu)
                    # 2. Native CUDA eigh
                    w, V = torch.linalg.eigh(mat_gpu)
                    # 3. Exponentiate
                    U = V @ torch.diag(torch.exp(-1j * w * dt)) @ V.mH
                    expm_cache[key] = U
            # else:
            #     print("Cache hit for gate at", where)
        gates[where] = U.reshape(4, 4, 4, 4).contiguous()

    return gates, expm_cache

def get_adjacent_gate_cached(dt, pxx, expm_cache=None, tol=0.0):
    r"""
    Builds the gate by vectorizing the 2-site Unitary Propagator.
    H2 = pxx * (X ⊗ X), U2 = exp(-i H2 dt).
    """
    # Create Pauli X on GPU
    sX = torch.tensor([[0, 1], [1, 0]], dtype=target_dtype, device=device)
    H2 = pxx * torch.kron(sX, sX)

    # Cache logic (use CPU bytes for key)
    H2_cpu = H2.cpu().numpy()
    dt_key = round(float(dt), 15)
    key = (dt_key, H2_cpu.tobytes())

    if expm_cache is None: expm_cache = {}

    U2 = expm_cache.get(key)
    if U2 is None:
        if dt_key == 0.0:
            U2 = torch.eye(H2.shape[0], dtype=target_dtype, device=device)
        else:
            # Native CUDA eigh
            w, V = torch.linalg.eigh(H2)
            U2 = V @ torch.diag(torch.exp(-1j * w * dt)) @ V.mH
        expm_cache[key] = U2

    # Vectorize: U (kron) U_conj
    U_vec = torch.kron(U2, torch.conj(U2))

    # Reshape
    U_ten = U_vec.reshape(2, 2, 2, 2, 2, 2, 2, 2)

    # PERMUTE (Replace np.transpose)
    # 0, 2, 1, 3, 4, 6, 5, 7
    U_ten = U_ten.permute(0, 2, 1, 3, 4, 6, 5, 7)

    return U_ten.reshape(4, 4, 4, 4).contiguous(), expm_cache

# ----------------------------
# Cached run step
# ----------------------------

def run_mpo_step_cached_4th(
    rho, duration, dt, L,
    pxx, pz, omega, alpha,
    site_i, pauli_op, max_bond,
    expm_cache=None,
):
    """
    4th-order Suzuki–Yoshida TEBD using cached expm gates.

    Uses S4(dt) = S2(w1*dt) S2(w0*dt) S2(w1*dt),
    where S2 is Strang splitting implemented as:
        jumper-half, odd-half, even-full, odd-half, jumper-half

    Notes:
    - Requires generating gates for scaled timesteps (including negative w0).
    - Keeps truncation at every gate via gate_split_.
    """
    # --- 1) setup state ---
    mps = rho.copy()
    mps = insert_MPS_site(mps, site_i)
    mps.cyclic = False
    max_bond_now = mps.max_bond()
    # --- 2) build H_local once ---
    H_local = build_hamiltonian_part(L, pxx, pz, omega, alpha, site_i, pauli_op)

    # TEBD sweep ordering will be based on keys of any gate dict
    # (same bonds for all dt scalings)
    # We'll compute it after creating the first gate set.

    # --- 3) options ---

    # if target_dtype == torch.complex64:
    #    split_opts = {'max_bond': max_bond, 'cutoff': 1e-6, 'renorm': True, 'absorb': 'right'}
    # else:
    #    split_opts = {'max_bond': max_bond, 'cutoff': 1e-12, 'renorm': True, 'absorb': 'right'}
    split_opts = {'max_bond': max_bond, 'cutoff': 1e-12, 'cutoff_mode': 'rel', 'renorm': True, 'absorb': 'both'}
    steps = int(duration / dt)
    if steps <= 0:
        return mps, expm_cache

    # jumper indices
    idx_L = site_i
    idx_M = site_i + 1
    idx_R = site_i + 2
    has_jumper = (idx_R < mps.L) and (idx_L >= 0)

    # --- 4) 4th-order coefficients (Yoshida) ---
    cbrt2 = 2.0 ** (1.0 / 3.0)
    w1 = 1.0 / (2.0 - cbrt2)
    w0 = -cbrt2 / (2.0 - cbrt2)
    coeffs = (w1, w0, w1)

    # --- helper: build (or fetch) gates for a given sub-dt ---
    # We need both full and half Strang gates at that substep.
    gate_sets = {}  # dt_sub -> (full, half, even_bonds, odd_bonds, jumper_half)


    def get_gate_sets(dt_sub):
        nonlocal expm_cache, has_jumper

        # Round dt_sub for stable dict keys
        dt_key = round(float(dt_sub), 15)

        if dt_key in gate_sets:
            return gate_sets[dt_key]

        full, expm_cache = generate_hamiltonian_gates_cached(
            H_local, dt_sub, site_i, expm_cache=expm_cache, tol=1e-15
        )
        half, expm_cache = generate_hamiltonian_gates_cached(
            H_local, dt_sub / 2.0, site_i, expm_cache=expm_cache, tol=1e-15
        )

        even_bonds = sorted([b for b in full if b[0] % 2 == 0])
        odd_bonds  = sorted([b for b in full if b[0] % 2 == 1])
        if has_jumper:
            jumper_half, expm_cache = get_adjacent_gate_cached(dt_sub / 2.0, pxx, expm_cache=expm_cache, tol=1e-15)
            gate_sets[dt_key] = (full, half, even_bonds, odd_bonds, jumper_half)
        else:
            gate_sets[dt_key] = (full, half, even_bonds, odd_bonds, None)

        return gate_sets[dt_key]

    # --- helper: one Strang (2nd order) step with sub-timestep dt_sub ---
    def strang_step(dt_sub):
        full, half, even_bonds, odd_bonds, jumper_half = get_gate_sets(dt_sub)

        # A) jumper block (symmetric half)
        if has_jumper:
            mps.gate_split_(SWAP_GATE_4, (idx_M, idx_R), **split_opts)
            mps.gate_split_(jumper_half, (idx_L, idx_M), **split_opts)
            mps.gate_split_(SWAP_GATE_4, (idx_M, idx_R), **split_opts)

        # B) chain evolution (Strang: odd half, even full, odd half)
        for bond in odd_bonds:
            mps.gate_split_(half[bond], bond, **split_opts)
        for bond in even_bonds:
            mps.gate_split_(full[bond], bond, **split_opts)
        for bond in odd_bonds:
            mps.gate_split_(half[bond], bond, **split_opts)

        # C) jumper block (symmetric half)
        if has_jumper:
            mps.gate_split_(SWAP_GATE_4, (idx_M, idx_R), **split_opts)
            mps.gate_split_(jumper_half, (idx_L, idx_M), **split_opts)
            mps.gate_split_(SWAP_GATE_4, (idx_M, idx_R), **split_opts)

    # --- 5) evolution loop: 4th order via 3 Strang steps ---
    for n in tqdm(range(steps)):
        strang_step(coeffs[0] * dt)
        strang_step(coeffs[1] * dt)  # negative substep
        strang_step(coeffs[2] * dt)
        if mps.max_bond() > max_bond_now:
            max_bond_now = mps.max_bond()
        curr_trace = get_vectorized_trace(mps)
        # if np.abs(1-curr_trace)>1e-2:
        #     print('Warning! Current trace:', curr_trace)
        mps /= curr_trace
        # if np.abs(1-curr_trace)>1e-2:
        #     print('Warning! Current trace after normalization:', get_vectorized_trace(mps))
        # should_log = (n % 20 == 0) or (n == steps - 1)
        # if should_log:
        #    print(f"    [DEBUG] Step {n+1}/{steps}")
        # print(f"    [DEBUG] Max Bond Dim: {mps.max_bond()}")
    # print(mps)
    mps = mps_trace_out_site_mpslevel(mps, site_i + 1)
    return mps, expm_cache, max_bond_now

In [7]:
# 5. Energy and Fidelity Preparation
# ============================================================

# ============================================================
# 1. Preparation Function (Uses 'q{i}' to avoid 'b{i}' bond collision)
# ============================================================
def prepare_rho_and_states(rho_vec, psi_targ, ham_op):

    # 1. Detect device/dtype from rho
    ref_tensor = list(rho_vec.tensors)[0].data
    dev = ref_tensor.device
    dtype = ref_tensor.dtype

    # 2. Convert sys_ground and Ham to match rho's device/dtype
    def to_compat(x):
        return torch.as_tensor(x).to(device=dev, dtype=dtype)

    psi_targ.apply_to_arrays(to_compat)
    ham_op.apply_to_arrays(to_compat)

    # 3. Manually reshape rho: Vectorized (d=4) -> MPO (d=2,2)
    rho_mpo = rho_vec.copy()

    for i in range(rho_mpo.L):
        t = rho_mpo[i]
        k_ind = f'k{i}'       # Current fused index (size 4)
        q_ind = f'q{i}'       # NEW Upper index (size 2).
                              # MUST use 'q' or similar, because 'b' collides with MPS bonds!

        new_inds_list = list(t.inds)

        if k_ind in new_inds_list:
            idx = new_inds_list.index(k_ind)
            old_shape = list(t.shape)

            # Split dimension 4 -> 2, 2
            new_shape = old_shape[:idx] + [2, 2] + old_shape[idx+1:]

            # Reshape data
            new_data = t.data.reshape(new_shape)

            # Update indices: replace 'k{i}' with 'k{i}', 'q{i}'
            new_inds_list[idx] = k_ind
            new_inds_list.insert(idx + 1, q_ind)

            t.modify(data=new_data, inds=new_inds_list)

    return rho_mpo, psi_targ, ham_op

# ============================================================
# 2. Calculation Function (Already correct, just ensures 'q' matching)
# ============================================================
def fid_energy_calculation(rho_mpo, sys_ground, Ham_MPO):

    # --- A. Fidelity: < psi | rho | psi > ---
    # Bra needs to have 'q' indices to match rho's upper 'q' indices.
    bra = sys_ground.H
    bra.reindex({f'k{i}': f'q{i}' for i in range(sys_ground.L)}, inplace=True)

    # Sandwich: Bra(q) ... Rho(q, k) ... Ket(k)
    fidelity_tn = bra & rho_mpo & sys_ground

    # Contract and extract scalar
    data1 = fidelity_tn.contract(all, optimize='auto-hq').item().real

    # --- B. Energy: Tr( rho * H ) ---
    # We want to contract Rho(k_up, q_down) with H(q_up, k_down).
    # (Note: MPO multiplication usually matches lower-to-upper)

    # Rho has indices: k{i} (down), q{i} (up)
    # We prepare H to have: q{i} (down), k{i} (up)
    # This closes the loop: Rho_up connects to H_down, H_up connects to Rho_down.

    H_trace = Ham_MPO.copy()
    rename_map = {}
    for i in range(H_trace.L):
        # Ham's original lower 'k' -> connects to Rho's upper 'q'
        rename_map[f'k{i}'] = f'q{i}'
        # Ham's original upper 'b' -> connects to Rho's lower 'k'
        rename_map[f'b{i}'] = f'k{i}'

    H_trace.reindex(rename_map, inplace=True)

    # Contract
    energy_tn = rho_mpo & H_trace
    data2 = energy_tn.contract(all, optimize='auto-hq').item().real

    return data1, data2

In [8]:
import scipy.linalg
import numpy as np
import torch
import functools
import autoray
import quimb.tensor as qtn
import random

# ==============================================================================
# RUNNER
# ==============================================================================

# ==============================================================================
# CONFIGURATION
# ==============================================================================
inputL = 4
inputbound = 50
L = inputL
max_bond = inputbound
random.seed(42)

J = 1.0
g = 1.5
pxx = -J
pz = g
sigma = 4
alpha = 0.1 / sigma**0.5
Ss = 4 * sigma
dTime = 0.01
BB = 5.0
Nstep = 500

#DMRG H
Ham_MPO = build_sys_hamiltonian(L, pxx, pz)
dmrg = qtn.DMRG2(Ham_MPO)
dmrg.solve(max_sweeps=80, verbosity=0, cutoffs=1e-10)
sys_ground = dmrg.state
sys_ground /= 1.0
minE=qtn.expec_TN_1D(sys_ground.H, Ham_MPO, sys_ground)
print("DMRG Ground state norm = ", sys_ground.H @ sys_ground)
print("DMRG Ground state energy = ", minE)

## ==============================================================================

print(f"--- Running Simulation (L={L}, Bond={max_bond}) ---")
vec_zero = to_device([1., 0., 0., 0.])
vec_I = to_device([1., 0., 0., 1.]) / (2)
rho_mps_initial = qtn.MPS_product_state(
    [vec_I] * L,           # List of L copies of vec_I
    tags='RHO_ID'
) # rho_initial=maximally mixed

rho_mps = rho_mps_initial.copy()
# 1. Detect the dtype and device from rho_mps
ref_tensor = list(rho_mps.tensors)[0].data
target_dtype = ref_tensor.dtype
target_device = ref_tensor.device
# 2. Define a converter that enforces this match
def to_compatible_torch(x):
    # Convert to tensor, then cast to the correct dtype and move to correct device
    return torch.as_tensor(np.array(x).copy()).to(dtype=target_dtype, device=target_device)

# 3. Apply the conversion
sys_ground.apply_to_arrays(to_compatible_torch)
Ham_MPO.apply_to_arrays(to_compatible_torch)

# cache setup
expm_cache = {}
max_cache_size = 4*L+2
data1 = np.zeros(Nstep)
data2 = np.zeros(Nstep)
for n in range(Nstep):
    omega = random.uniform(0, BB)
    # site_i = random.choice([-1, L-1]) #boundary: bath before site 0 or after site L-1
    site_i = random.randint(0, L-1) #bulk: bath after site_i
    sign = random.choice([-1, 1])
    pauli_op = random.randint(1, 3)
    print(f"=== Testing Insertion at Step={n} === site_i={site_i} === pauli_op={pauli_op} === omega={omega:.4f} ===")
    # mpo simulation
    dim = 2**(L)
    with CodeTimer("Time Evolution"):
        # rho_mps, max_bond_now = run_mpo_step_boundary_TEBD(
        #     rho_mps, duration=2*Ss, dt=dTime, L=L,
        #     pxx=pxx, pz=pz, omega=omega, alpha=sign*alpha,
        #     site_i=site_i, pauli_op=pauli_op, max_bond=max_bond
        # ) #for boundary sites
        rho_mps, expm_cache, max_bond_now = run_mpo_step_cached_4th(
            rho_mps, duration=2*Ss, dt=dTime, L=L,
            pxx=pxx, pz=pz, omega=omega, alpha=sign*alpha,
            site_i=site_i, pauli_op=pauli_op, max_bond=max_bond,
            expm_cache=expm_cache
        )
    if len(expm_cache) > max_cache_size: #this should not happen
        print("Cache size exceeded, clearing oldest entry.")
        print("Current cache size:", len(expm_cache))
        evicted_key, _ = expm_cache.popitem(last=False)
    # fidelity and energy calculation
    with CodeTimer("Measure Calc"):
        rho_mpo, sys_ground, Ham_MPO = prepare_rho_and_states(rho_mps, sys_ground, Ham_MPO)
    data1[n], data2[n] = fid_energy_calculation(rho_mpo, sys_ground, Ham_MPO)
    print(f"Step = {n} MPO measure, Fidelity = {data1[n]:.6f}, Energy = {data2[n]:.6f}, Max Bond Dim: {max_bond_now}")

DMRG Ground state norm =  0.9999999999999997
DMRG Ground state energy =  -6.50389155712641
--- Running Simulation (L=4, Bond=50) ---
=== Testing Insertion at Step=0 === site_i=0 === pauli_op=1 === omega=3.1971 ===


100%|██████████| 3200/3200 [00:20<00:00, 157.90it/s]


  [TIMER] Time Evolution: 20.40914 sec
  [TIMER] Measure Calc: 0.00045 sec
Step = 0 MPO measure, Fidelity = 0.062936, Energy = -0.010503, Max Bond Dim: 16
=== Testing Insertion at Step=1 === site_i=0 === pauli_op=3 === omega=1.1161 ===


100%|██████████| 3200/3200 [00:20<00:00, 157.01it/s]


  [TIMER] Time Evolution: 20.38470 sec
  [TIMER] Measure Calc: 0.00012 sec
Step = 1 MPO measure, Fidelity = 0.062938, Energy = -0.267509, Max Bond Dim: 16
=== Testing Insertion at Step=2 === site_i=0 === pauli_op=1 === omega=2.1096 ===


...

  [TIMER] Time Evolution: 19.25067 sec
  [TIMER] Measure Calc: 0.00011 sec
Step = 497 MPO measure, Fidelity = 0.984459, Energy = -6.468345, Max Bond Dim: 16
=== Testing Insertion at Step=498 === site_i=0 === pauli_op=2 === omega=0.0463 ===


100%|██████████| 3200/3200 [00:18<00:00, 177.36it/s]


  [TIMER] Time Evolution: 18.04469 sec
  [TIMER] Measure Calc: 0.00017 sec
Step = 498 MPO measure, Fidelity = 0.983757, Energy = -6.466239, Max Bond Dim: 16
=== Testing Insertion at Step=499 === site_i=3 === pauli_op=3 === omega=2.2717 ===


100%|██████████| 3200/3200 [00:09<00:00, 350.52it/s]

  [TIMER] Time Evolution: 9.13167 sec
  [TIMER] Measure Calc: 0.00011 sec
Step = 499 MPO measure, Fidelity = 0.983746, Energy = -6.466718, Max Bond Dim: 16
